# Alchemy GeomML + TDA — Quickstart

Полный пайплайн: загрузка данных → обучение → оценка → AutoML.

**Датасет:** Alchemy v20191129 (202,579 молекул, 12 квантово-механических свойств)

**Модели:** FCNN, SchNet, EGNN, EGNN+TDA, EGNN Vector, EGNN Vector+TDA

## 1. Установка зависимостей

Если запуск локально (не в Kaggle/Colab):

In [ ]:
!pip install -q torch torch-geometric gudhi rdkit egnn-pytorch pandas matplotlib seaborn pyyaml scikit-learn

## 2. Клонирование репозитория и загрузка данных

In [ ]:
import os
if not os.path.exists('alchemy-geom-tda'):
    !git clone https://github.com/ThomasMoore25/alchemy-geom-tda.git
%cd alchemy-geom-tda
!git log --oneline -1

In [ ]:
# Скачать датасет Alchemy (136 МБ zip, ~600 МБ распакованный)
!python data/download_alchemy.py 2>&1 | tail -10

## 3. Проверка окружения

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
import torch_geometric
import egnn_pytorch
import rdkit
import gudhi

print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Device count: {torch.cuda.device_count()}')
print(f'torch-geometric: {torch_geometric.__version__}')
print(f'rdkit: {rdkit.__version__}')
print(f'gudhi: {gudhi.__version__}')

## 4. Smoke-тест (5 минут на CPU, проверка установки)

Обучаем EGNN на 100 молекулах, 3 эпохи. Должен быть виден убывающий loss.

In [ ]:
import sys, os
sys.path.insert(0, 'src')
os.makedirs('results/smoke', exist_ok=True)

sys.argv = ['train.py',
    '--model', 'egnn',
    '--target', 'all',
    '--epochs', '3',
    '--max_train', '100',
    '--max_val', '20',
    '--max_test', '20',
    '--batch_size', '32',
    '--device', 'cpu',
    '--output_dir', 'results/smoke',
]

from train import main as train_main
train_main()

## 5. Полное обучение одной модели (GPU, ~3 часа)

EGNN на полном датасете (202k молекул), 100 эпох с EarlyStopping.

In [ ]:
# Раскомментировать для полного запуска (нужен GPU)
# sys.argv = ['train.py',
#     '--model', 'egnn',
#     '--target', 'all',
#     '--epochs', '100',
#     '--batch_size', '512',
#     '--device', 'cuda',
#     '--num_workers', '4',
#     '--output_dir', 'results/experiments/batch_size_512',
# ]
# from train import main as train_main
# train_main()

## 6. Обучение всех 6 моделей (run_all)

FCNN, SchNet, EGNN, EGNN+TDA, EGNN Vector, EGNN Vector+TDA — последовательно.

In [ ]:
# Раскомментировать для полного запуска (4-8 часов на GPU)
# sys.argv = ['run_all.py',
#     '--models', 'all',
#     '--epochs', '100',
#     '--batch_size', '512',
#     '--device', 'cuda',
#     '--num_workers', '4',
#     '--multi_gpu',  # если 2+ GPU
# ]
# from run_all import main as run_all_main
# run_all_main()

## 7. Графики обучения

Кривые train/val loss и MAE по эпохам + parity plots.

In [ ]:
import sys
sys.path.insert(0, 'src')
from plot import plot_training_history, plot_parity, compare_histories

# Укажите путь к history CSV (замените на актуальный)
csv_path = 'results/smoke/history_egnn_all_20260717_091245.csv'  # smoke-test
# csv_path = 'results/experiments/batch_size_512/history_egnn_all_20260712_160620.csv'  # полный

if os.path.exists(csv_path):
    plot_training_history(csv_path, show=True)
    plot_parity(csv_path, show=True)
else:
    print(f'CSV не найден: {csv_path}')
    print('Доступные history CSV:')
    import glob
    for f in sorted(glob.glob('results/**/history_*.csv', recursive=True)):
        print(f'  {f}')

## 8. Robustness evaluation

Оценка устойчивости обученной модели к шуму в координатах (σ ∈ {0.0, 0.05, 0.10, 0.15}).

In [ ]:
# Раскомментировать после обучения модели
# sys.argv = ['eval_robustness.py',
#     '--model', 'egnn', '--target', 'all',
#     '--checkpoint', 'checkpoints/egnn_all_best.pt',
#     '--target_stats', 'results/experiments/batch_size_512/target_stats_egnn_all.json',
#     '--noise_sigma', '0.0,0.05,0.10,0.15',
#     '--device', 'cuda',
# ]
# from eval_robustness import main as eval_main
# eval_main()

## 9. AutoML — автоматический выбор архитектуры (программа максимум)

Извлекает геометрические priors из TDA и рекомендует оптимальную архитектуру.

In [ ]:
sys.argv = ['select.py',
    '--data_dir', 'data/alchemy',
    '--n_molecules', '30',
    '--threshold', '0.95',
    '--output_json', 'results/automl/recommendation.json',
]

import sys
sys.path.insert(0, 'src/automl')
from select import main as automl_main
automl_main()

## 10. AutoML с quick-train сравнением

Дополнительно quick-train несколько моделей и сравнить val_loss.

In [ ]:
# Раскомментировать для запуска (10-30 минут на GPU)
# sys.argv = ['select.py',
#     '--data_dir', 'data/alchemy',
#     '--n_molecules', '500',
#     '--epochs', '3',
#     '--quick_train',
#     '--candidates', 'fcnn,schnet,egnn,egnn_tda',
#     '--device', 'cuda',
#     '--output_json', 'results/automl/recommendation_full.json',
# ]
# sys.path.insert(0, 'src/automl')
# from select import main as automl_main
# automl_main()

## 11. Просмотр отчёта AutoML

In [ ]:
import json
from pathlib import Path

report_path = 'results/automl/recommendation.json'
if Path(report_path).exists():
    with open(report_path) as f:
        report = json.load(f)
    print('=== AutoML Report ===')
    print(f'Timestamp: {report["timestamp"]}')
    print(f'\nPriors (n={report["priors"]["n_molecules_success"]} molecules):')
    for k, v in report['priors'].items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')
    print(f'\nRecommendation:')
    for k, v in report['recommendation'].items():
        print(f'  {k}: {v}')
else:
    print(f'Отчёт не найден: {report_path}')
    print('Запустите ячейку 9 сначала.')

## 12. Тесты (проверка кода)

In [ ]:
!python -m pytest --tb=short 2>&1 | tail -10

## 13. Структура репозитория

In [ ]:
!find . -maxdepth 2 -type f -name '*.py' -o -name '*.md' -o -name '*.yml' -o -name '*.yaml' -o -name '*.txt' -o -name 'Dockerfile' 2>/dev/null | grep -v __pycache__ | grep -v .git | sort | head -40

## Что дальше

1. **Полное обучение на GPU:** раскомментируйте ячейки 5-6, установите `--device cuda`.
2. **Robustness eval:** после обучения прогоните ячейку 8 на каждой модели.
3. **AutoML с quick_train:** ячейка 10 для численного сравнения архитектур.
4. **Публикация:** загрузите чекпойнты + history CSV на GitHub через Git LFS.

См. `QUICKSTART.md` для краткой справки по CLI.